# 02 — PINN for PDE (Heat Equation)

**Module 3 · Week 6 · TA-3**

This notebook covers:
- Extending PINN from ODE to PDE
- Spatiotemporal input `(x, t)`
- Computing partial derivatives `u_t`, `u_x`, `u_xx`
- Enforcing IC and BC
- Comparing with analytic solution via heatmap

**Target PDE:** 1D Heat Equation
```
∂u/∂t = α · ∂²u/∂x²,   x ∈ [0,1],   t ∈ [0,1]
BC:  u(0,t) = 0,   u(1,t) = 0
IC:  u(x,0) = sin(πx)
Analytic: u(x,t) = sin(πx) · exp(-α·π²·t),   α = 0.01
```

---

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
alpha = 0.01  # thermal diffusivity
print(f'Device: {device}')

## 1. Analytic Solution (Reference)

In [ ]:
def analytic_solution(x, t, alpha):
    return np.sin(np.pi * x) * np.exp(-alpha * np.pi**2 * t)

# Visualize on a grid
x_grid = np.linspace(0, 1, 100)
t_grid = np.linspace(0, 1, 100)
X, T = np.meshgrid(x_grid, t_grid)
U_ref = analytic_solution(X, T, alpha)

plt.figure(figsize=(6, 4))
plt.contourf(X, T, U_ref, levels=50, cmap='hot')
plt.colorbar(label='u(x,t)')
plt.xlabel('x')
plt.ylabel('t')
plt.title('Reference: u(x,t) = sin(πx)·exp(-α π²t)')
plt.show()

## 2. Define PINN

In [ ]:
class PINN(nn.Module):
    def __init__(self, hidden: int = 64, n_layers: int = 4):
        super().__init__()
        # --- TODO: Define MLP ---
        # Input: 2 (x, t)
        # Hidden: n_layers layers of `hidden` units, tanh activation
        # Output: 1 (u(x,t))
        pass

    def forward(self, xt):
        pass

model = PINN().to(device)

## 3. Sample Training Points

In [ ]:
N_COLLOC = 2000  # interior collocation points
N_BC = 200       # boundary condition points (each boundary)
N_IC = 200       # initial condition points

# Interior: x in (0,1), t in (0,1)
x_c = torch.rand(N_COLLOC, 1)
t_c = torch.rand(N_COLLOC, 1)
xt_colloc = torch.cat([x_c, t_c], dim=1).to(device).requires_grad_(True)

# BC: x=0 and x=1 for t in [0,1]
t_bc = torch.rand(N_BC, 1)
xt_bc0 = torch.cat([torch.zeros(N_BC, 1), t_bc], dim=1).to(device)  # x=0
xt_bc1 = torch.cat([torch.ones(N_BC, 1), t_bc], dim=1).to(device)   # x=1

# IC: t=0, u=sin(pi*x)
x_ic = torch.rand(N_IC, 1)
xt_ic = torch.cat([x_ic, torch.zeros(N_IC, 1)], dim=1).to(device)
u_ic = torch.sin(np.pi * x_ic).to(device)

print(f'Collocation: {xt_colloc.shape}')
print(f'BC (x=0): {xt_bc0.shape}')
print(f'IC: {xt_ic.shape}')

## 4. PDE Residual

In [ ]:
def pde_residual(model, xt, alpha):
    """Compute heat equation residual: u_t - alpha * u_xx = 0."""
    u = model(xt)

    # --- TODO: Compute u_t and u_xx using autograd ---
    # Hint: use create_graph=True for both, allow_unused may be needed
    # grads = torch.autograd.grad(u, xt, grad_outputs=torch.ones_like(u),
    #                              create_graph=True)[0]
    # u_x = grads[:, 0:1]
    # u_t = grads[:, 1:2]
    # u_xx = torch.autograd.grad(u_x, xt, ...)[0][:, 0:1]
    # residual = u_t - alpha * u_xx
    pass

## 5. Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
N_STEPS = 20000
W_PDE = 1.0
W_BC = 10.0
W_IC = 10.0

loss_history = []

for step in range(N_STEPS):
    optimizer.zero_grad()

    # --- TODO: Compute total loss ---
    # 1. PDE residual loss at collocation points
    # 2. BC loss: model(xt_bc0) should be 0, model(xt_bc1) should be 0
    # 3. IC loss: MSE(model(xt_ic), u_ic)

    if (step + 1) % 2000 == 0:
        print(f'Step {step+1}/{N_STEPS}  Loss: {loss_history[-1]:.6f}')

## 6. Evaluate: Heatmap Comparison

In [ ]:
x_eval = torch.linspace(0, 1, 100)
t_eval = torch.linspace(0, 1, 100)
X_ev, T_ev = torch.meshgrid(x_eval, t_eval, indexing='xy')
xt_eval = torch.stack([X_ev.flatten(), T_ev.flatten()], dim=1).to(device)

with torch.no_grad():
    U_pred = model(xt_eval).cpu().numpy().reshape(100, 100)

U_ref_torch = analytic_solution(X_ev.numpy(), T_ev.numpy(), alpha)
error = np.abs(U_pred - U_ref_torch)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title in zip(axes,
    [U_ref_torch, U_pred, error],
    ['Reference', 'PINN Prediction', 'Absolute Error']):
    im = ax.contourf(X_ev.numpy(), T_ev.numpy(), data, levels=50, cmap='hot')
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.set_title(title)
plt.tight_layout()
plt.show()

print(f'Max absolute error: {error.max():.4f}')
print(f'Mean absolute error: {error.mean():.4f}')

## 7. Collocation Point Experiment

*(Complete as part of Week 6 assignment)*

Rerun with `N_COLLOC ∈ [100, 500, 2000, 5000]` and record:

| N_COLLOC | Final PDE loss | Max absolute error |
|----------|---------------|--------------------|